# Immune Data Embeddings

In [1]:
import sys
sys.path.append("..")

from models.scimmune.config import ScImmuneConfig
from models.scimmune.model import ScImmuneModel
from models.scimmune.tokenizer import ScImmuneTokenizer # refactored version

import torch
import os
import shutil
from utils.model_fns import generate_metadata_embeddings, generate_metadata_tokens, assign_ontology_embeddings
from gensim.models import Word2Vec
import anndata as ad
import pandas as pd
import scanpy as sc
import json

from pathlib import Path

/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Set folder paths
DATA_PATH = Path("../data/cellxgene_data")
ONTOLOGY_PATH = Path("../data/ontologies")
MODEL_PATH = Path("../models/scimmune")
UTIL_PATH = Path("../utils")

## Modify tokens and embeddings

In [3]:
shutil.copy(f"{MODEL_PATH}/config.json", f"{MODEL_PATH}/scimmune-model/config.json")
shutil.copy(f"{MODEL_PATH}/og_model.bin", f"{MODEL_PATH}/scimmune-model/pytorch_model.bin")

'../models/scimmune/scimmune-model/pytorch_model.bin'

In [4]:
local_config = ScImmuneConfig.from_pretrained(f"{MODEL_PATH}/scimmune-model") # load config locally
local_model = ScImmuneModel(local_config) # load model locally
local_tokenizer = ScImmuneTokenizer(vocab_file=f"{MODEL_PATH}/vocab_w_cell_type.json") # initialize tokenizer

In [5]:
len(local_tokenizer) # 350 new metadata tokens -> 60698 + 350 = 61048
new_vocab_len = len(local_tokenizer)

In [6]:
new_vocab_len

60838

In [7]:
local_model.resize_token_embeddings(new_vocab_len)

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(60838, 512, padding_idx=0)

In [8]:
local_model # inspect full model

ScImmuneModel(
  (gene_encoder): GeneEncoder(
    (embedding): Embedding(60838, 512, padding_idx=0)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (lin1): Linear(in_features=1, out_features=512, bias=True)
    (act): ReLU()
    (lin2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (no

## Re-initialize metadata tokens with Node2Vec vectors

In [9]:
embedding_layer = local_model.get_input_embeddings()
embedding_layer

Embedding(60838, 512, padding_idx=0)

In [14]:
# Set Node2Vec model folder path
n2vmodel_immune_path = ONTOLOGY_PATH / "node2vec_models" / "immune_cl_node2vec.model"
immune_cl_embeddings = Word2Vec.load(str(n2vmodel_immune_path)) # cell type

In [ ]:
# # Load all embedding vectors
# doid_embeddings = Word2Vec.load(f"{n2vmodel_path}/doid_node2vec.model") # disease
# cl_embeddings = Word2Vec.load(f"{n2vmodel_path}/cl_node2vec.model") # cell type
# hancestro_embeddings = Word2Vec.load(f"{n2vmodel_path}/hancestro_node2vec.model") # ethinicity
# hsapdv_embeddings = Word2Vec.load(f"{n2vmodel_path}/hsapdv_node2vec.model") # human development
# pato_embeddings = Word2Vec.load(f"{n2vmodel_path}/pato_node2vec.model") # sex
# uberon_embeddings = Word2Vec.load(f"{n2vmodel_path}/uberon_node2vec.model") # tissue

In [15]:
with open("vocab_w_cell_type.json", "r") as f:
    vocab_with_metadata_dict = json.load(f) # load this as a dict for lookup

FileNotFoundError: [Errno 2] No such file or directory: 'vocab_w_cell_type.json'

In [10]:
tag2ontology_map = {
    "disease" : "doid",
    "cell_type" : "cl",
    "self_reported_ethnicity" : "hancestro",
    "development_stage" : "hsapdv",
    "sex" : "pato",
    "tissue_general" : "uberon",
}

In [16]:
# Assign cell type ontology embeddings
assign_ontology_embeddings(
        tokenizer=local_tokenizer,
        model=local_model,
        node2vec_model_path=str(n2vmodel_immune_path),
        tag="cell_type"
        )

[INFO] Assigned 140 Node2Vec embeddings for tag <cell_type=...>


In [17]:
immune_cl_embeddings

In [19]:
# Check at random 
# doid_embeddings.wv["DOID:4"]
token = "<cell_type=CL:0000906>"
oid = token[1:-1].split("=")[-1] # get OID
token_id = local_tokenizer.convert_tokens_to_ids(token)

print(token)
print(token_id)

n2v_embedding = immune_cl_embeddings.wv[oid]

model_embedding = embedding_layer.weight.data[token_id]

<cell_type=CL:0000906>
60834


In [20]:
model_embedding

tensor([-9.4560e-02,  3.0770e-01, -6.1557e-02,  1.7030e-01,  3.3536e-01,
         3.3632e-01, -7.7996e-02, -4.3433e-02,  2.9867e-03,  1.1118e-01,
         1.4058e-01, -2.3965e-01,  6.8719e-01, -2.4565e-01, -9.7904e-02,
         3.0574e-01, -3.5656e-01, -2.6897e-01, -2.3866e-03, -2.1867e-01,
         1.9978e-01, -2.4935e-01,  1.1565e-01, -2.9013e-01, -4.6356e-01,
         1.0493e-01,  4.5777e-02,  2.8011e-01, -8.5724e-02,  2.4231e-01,
         2.5008e-01, -7.0555e-02,  1.0197e-01,  1.7142e-01,  2.1473e-02,
         2.7215e-02,  2.7078e-01,  8.9355e-02, -1.4867e-01, -3.0876e-01,
         1.0785e-01,  3.9159e-02,  9.2567e-02,  6.5590e-02,  1.9842e-01,
        -2.4493e-01, -1.0217e-01,  8.3739e-02,  5.2287e-01, -6.6123e-03,
        -3.0784e-01,  6.2610e-02, -1.7336e-01,  3.3002e-01, -3.1517e-01,
        -2.7534e-01,  2.7649e-01, -5.3647e-02,  8.2566e-02,  3.6684e-01,
         7.6349e-02,  3.8342e-01,  1.2487e-01,  2.3971e-02,  1.0912e-01,
         1.7955e-01,  3.1795e-02,  5.6299e-02, -3.3

In [21]:
n2v_embedding

array([-9.45604816e-02,  3.07695627e-01, -6.15573674e-02,  1.70298457e-01,
        3.35355610e-01,  3.36317807e-01, -7.79959038e-02, -4.34329398e-02,
        2.98667233e-03,  1.11181132e-01,  1.40575230e-01, -2.39652276e-01,
        6.87185645e-01, -2.45650366e-01, -9.79036540e-02,  3.05743873e-01,
       -3.56558174e-01, -2.68966109e-01, -2.38659512e-03, -2.18668878e-01,
        1.99783131e-01, -2.49349609e-01,  1.15653083e-01, -2.90125906e-01,
       -4.63562518e-01,  1.04925834e-01,  4.57774773e-02,  2.80109227e-01,
       -8.57236460e-02,  2.42309272e-01,  2.50080764e-01, -7.05551207e-02,
        1.01972155e-01,  1.71420157e-01,  2.14725602e-02,  2.72150934e-02,
        2.70778805e-01,  8.93548876e-02, -1.48665994e-01, -3.08759004e-01,
        1.07852779e-01,  3.91592793e-02,  9.25671011e-02,  6.55901060e-02,
        1.98422462e-01, -2.44930029e-01, -1.02168895e-01,  8.37394074e-02,
        5.22867322e-01, -6.61232788e-03, -3.07839274e-01,  6.26099408e-02,
       -1.73361450e-01,  

In [22]:
local_model.save_pretrained("scImmune_metadata_model")